# Qubits on the Bloch sphere: gates, shots and entanglement

**Professor Nano Labs · Quantum computing**

**Try it live in your browser:** [Qubits on the Bloch Sphere lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/qubits-bloch-sphere/)
**Watch first:** [Mapping the qubit state onto the Bloch Sphere](https://www.youtube.com/watch?v=lqWSziZJsLs)
**Watch later:** [Single qubit and its logic gates](https://www.youtube.com/watch?v=rD_fH7O-D5Y) · [Two qubit state: Separable vs Entangled vs Bell states](https://www.youtube.com/watch?v=4BRQsrh4oGw) · [Two and three qubits quantum gates](https://www.youtube.com/watch?v=iMjpZwISIlA) · [Quantum entanglement; exposing its key features](https://www.youtube.com/watch?v=9vDyS2_Hufk)

A qubit's pure state is a superposition of $|0\rangle$ and $|1\rangle$. Normalization and an overall phase, which no measurement can detect, leave two angles, so every state is a point on a unit sphere:

$$|\psi\rangle=\cos\frac{\theta}{2}\,|0\rangle+e^{i\varphi}\sin\frac{\theta}{2}\,|1\rangle
\quad\longleftrightarrow\quad
\vec r=(\sin\theta\cos\varphi,\ \sin\theta\sin\varphi,\ \cos\theta)=\langle\psi|\vec\sigma|\psi\rangle .$$

This notebook does the same calculations as the browser lab, with NumPy:

1. states, angles and the Bloch vector;
2. single-qubit gates as rotations of the sphere;
3. a circuit drawn on the sphere, including the path *during* each gate;
4. measurement shots and state tomography;
5. two qubits: CNOT, Bell states, partial traces and the concurrence.

It needs only NumPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

# Pauli matrices and the identity
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
PAULI = (X, Y, Z)

## 1. States, angles and the Bloch vector

`ket(theta, phi)` builds the state from its two angles, and `bloch(psi)` returns $\vec r=\langle\psi|\vec\sigma|\psi\rangle$. The six "cardinal" states sit at the ends of the three axes: $|0\rangle,|1\rangle$ on z, $|\pm\rangle=(|0\rangle\pm|1\rangle)/\sqrt2$ on x and $|\pm i\rangle=(|0\rangle\pm i|1\rangle)/\sqrt2$ on y.

In [ ]:
def ket(theta, phi):
    # cos(theta/2)|0> + e^{i phi} sin(theta/2)|1>
    return np.array([np.cos(theta / 2), np.exp(1j * phi) * np.sin(theta / 2)])

def bloch(psi):
    # <psi|sigma|psi> for a normalized one-qubit state
    return np.real([psi.conj() @ s @ psi for s in PAULI])

def angles(r):
    # polar angle theta and azimuth phi (degrees) of a Bloch vector
    theta = np.degrees(np.arccos(np.clip(r[2] / np.linalg.norm(r), -1, 1)))
    phi = np.degrees(np.arctan2(r[1], r[0])) % 360
    return theta, phi

CARDINAL = {'|0>': (0, 0), '|1>': (180, 0), '|+>': (90, 0),
            '|->': (90, 180), '|+i>': (90, 90), '|-i>': (90, 270)}
print(f"{'state':>6}   theta   phi    Bloch vector (x, y, z)")
for name, (th, ph) in CARDINAL.items():
    r = bloch(ket(np.radians(th), np.radians(ph)))
    print(f"{name:>6}   {th:5.0f}  {ph:4.0f}    {r}")

# A general state: the Bloch vector is the spherical-coordinate unit vector
th, ph = np.radians(60), np.radians(30)
r = bloch(ket(th, ph))
expected = [np.sin(th) * np.cos(ph), np.sin(th) * np.sin(ph), np.cos(th)]
print('\ntheta = 60 deg, phi = 30 deg ->', r, '| matches formula:', np.allclose(r, expected))
print('P(0) =', abs(ket(th, ph)[0])**2, ' = (1 + z)/2 =', (1 + r[2]) / 2)

## 2. Every single-qubit gate is a rotation

A rotation by angle $\alpha$ about the unit axis $\hat n$ is

$$R_{\hat n}(\alpha)=e^{-i\alpha\,\hat n\cdot\vec\sigma/2}=\cos\frac{\alpha}{2}\,I-i\sin\frac{\alpha}{2}\,\hat n\cdot\vec\sigma .$$

Every named gate is an overall phase times one of these. The lab stores each gate as $(\gamma,\hat n,\alpha)$ with $U=e^{i\gamma}R_{\hat n}(\alpha)$; the cell below checks those numbers against the usual matrices. It also checks that $U$ really turns the Bloch vector by $\alpha$ about $\hat n$ (Rodrigues' rotation formula), and that a full turn gives $-I$, not $I$.

In [ ]:
def R(n, alpha):
    n = np.asarray(n, float) / np.linalg.norm(n)
    return np.cos(alpha / 2) * I2 - 1j * np.sin(alpha / 2) * (n[0] * X + n[1] * Y + n[2] * Z)

AXES = {'x': [1, 0, 0], 'y': [0, 1, 0], 'z': [0, 0, 1], 'x+z': [1, 0, 1]}
s2 = 1 / np.sqrt(2)
GATES = {   # name: (matrix, gamma, axis, alpha)
    'X':  (X, np.pi / 2, 'x', np.pi),
    'Y':  (Y, np.pi / 2, 'y', np.pi),
    'Z':  (Z, np.pi / 2, 'z', np.pi),
    'H':  (s2 * np.array([[1, 1], [1, -1]], dtype=complex), np.pi / 2, 'x+z', np.pi),
    'S':  (np.diag([1, 1j]), np.pi / 4, 'z', np.pi / 2),
    'S†': (np.diag([1, -1j]), -np.pi / 4, 'z', -np.pi / 2),
    'T':  (np.diag([1, np.exp(1j * np.pi / 4)]), np.pi / 8, 'z', np.pi / 4),
    'T†': (np.diag([1, np.exp(-1j * np.pi / 4)]), -np.pi / 8, 'z', -np.pi / 4),
}

def rodrigues(r, n, alpha):
    # rotate the 3-vector r by alpha about the axis n
    n = np.asarray(n, float) / np.linalg.norm(n)
    return r * np.cos(alpha) + np.cross(n, r) * np.sin(alpha) + n * (n @ r) * (1 - np.cos(alpha))

rng = np.random.default_rng(1)
print('gate   gamma    axis   alpha    |U - e^{i gamma} R| , max Bloch-rotation error')
for name, (U, gamma, axis, alpha) in GATES.items():
    err_U = np.abs(U - np.exp(1j * gamma) * R(AXES[axis], alpha)).max()
    err_r = 0
    for _ in range(200):     # random input states
        psi = ket(np.arccos(rng.uniform(-1, 1)), rng.uniform(0, 2 * np.pi))
        err_r = max(err_r, np.abs(bloch(U @ psi) - rodrigues(bloch(psi), AXES[axis], alpha)).max())
    print(f"{name:>3}   {np.degrees(gamma):6.1f}   {axis:>4}   {np.degrees(alpha):6.1f}   {err_U:.1e}   {err_r:.1e}")

print('\nR_x(360 deg) =\n', R(AXES['x'], 2 * np.pi), '\n  -> -I: the arrow comes back, the amplitudes change sign.')
print('R_x(720 deg) =\n', R(AXES['x'], 4 * np.pi))

## 3. A circuit on the sphere, including the path during each gate

To animate a gate, the lab follows $U(s)=e^{i\gamma s}R_{\hat n}(\alpha s)$ for $0\le s\le 1$. For the named gates this is the **principal fractional power** $U^s=e^{s\log U}$, the state partway through a gate pulse. Below, `frac_power` computes $U^s$ independently by diagonalizing $U$, and the plots show the circuit $|0\rangle\to H\to T\to H$ on the sphere and as Bloch components against position in the circuit.

In [ ]:
def frac_power(U, s):
    # principal U^s via eigendecomposition (U is unitary, so it is diagonalizable)
    w, V = np.linalg.eig(U)
    return V @ np.diag(np.exp(s * np.log(w.astype(complex)))) @ np.linalg.inv(V)

# the decomposition and the eigendecomposition give the same path
worst = 0
for name, (U, gamma, axis, alpha) in GATES.items():
    for s in np.linspace(0, 1, 21):
        worst = max(worst, np.abs(frac_power(U, s) - np.exp(1j * gamma * s) * R(AXES[axis], alpha * s)).max())
print('max |U^s - e^{i gamma s} R(alpha s)| over all gates and s:', f'{worst:.1e}')

def run_circuit(psi0, names, per=60):
    # Bloch vectors sampled along the circuit; returns positions t and vectors r
    ts, rs, psi = [0.0], [bloch(psi0)], psi0
    for k, name in enumerate(names):
        U = GATES[name][0]
        for s in np.linspace(0, 1, per + 1)[1:]:
            ts.append(k + s)
            rs.append(bloch(frac_power(U, s) @ psi))
        psi = U @ psi
    return np.array(ts), np.array(rs), psi

def draw_sphere(ax):
    u, v = np.linspace(0, 2 * np.pi, 40), np.linspace(0, np.pi, 20)
    ax.plot_wireframe(np.outer(np.cos(u), np.sin(v)), np.outer(np.sin(u), np.sin(v)),
                      np.outer(np.ones_like(u), np.cos(v)), color='0.85', lw=0.4)
    for a in np.eye(3):
        ax.plot(*np.c_[-a, a], color='0.6', lw=0.8)
    for pos, lab in [((0, 0, 1.2), '|0>'), ((0, 0, -1.25), '|1>'), ((1.25, 0, 0), '|+>'),
                     ((-1.25, 0, 0), '|->'), ((0, 1.25, 0), '|+i>'), ((0, -1.3, 0), '|-i>')]:
        ax.text(*pos, lab, ha='center', va='center')
    ax.set_box_aspect((1, 1, 1)); ax.set_axis_off()
    ax.view_init(elev=16, azim=28)

circuit = ['H', 'T', 'H']
ts, rs, psi_end = run_circuit(ket(0, 0), circuit)

fig = plt.figure(figsize=(11, 4.6))
ax = fig.add_subplot(1, 2, 1, projection='3d')
draw_sphere(ax)
ax.plot(*rs.T, color='#5b45b0', lw=2)
ax.quiver(0, 0, 0, *rs[-1], color='#5b45b0', lw=2.5, arrow_length_ratio=0.12)
ax.set_title('|0> -> H -> T -> H on the Bloch sphere')

ax2 = fig.add_subplot(1, 2, 2)
for k, (lab, col) in enumerate([('x', '#1f6fb4'), ('y', '#e07b1f'), ('z', '#5b45b0')]):
    ax2.plot(ts, rs[:, k], color=col, lw=2, label=lab)
for k in range(1, len(circuit)):
    ax2.axvline(k, color='0.75', ls=':')
for k, name in enumerate(circuit):
    ax2.text(k + 0.5, 1.08, name, ha='center')
ax2.set_xlabel('Position in the circuit (gates)'); ax2.set_ylabel('Bloch component')
ax2.set_ylim(-1.15, 1.2); ax2.legend(loc='lower left'); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('final Bloch vector:', bloch(psi_end), ' P(0) =', abs(psi_end[0])**2)

## 4. Measurement: shots, not arrows

Measuring along an axis $\hat n$ gives +1 or −1 with probabilities $P(\pm)=(1\pm\vec r\cdot\hat n)/2$, and each shot needs a fresh copy of the state. The running fraction of +1 outcomes settles on $P(+)$ with a scatter of about $\sqrt{P(1-P)/N}$.

Counting along x, y and z gives the three components $\hat r_n=(N_+-N_-)/N$, which rebuilds the arrow: **state tomography**. Its error falls like $1/\sqrt N$.

In [ ]:
rng = np.random.default_rng(7)

def shots(r, n_axis, N):
    # N outcomes (+1 or -1) along the unit axis n_axis
    p_plus = (1 + r @ n_axis) / 2
    return np.where(rng.random(N) < p_plus, 1, -1)

r = bloch(ket(np.radians(60), 0))      # theta = 60 deg, phi = 0 (Question 4 in the lab)
print('Bloch vector:', r, ' P(0) along z =', (1 + r[2]) / 2, ' P(+) along x =', (1 + r[0]) / 2)

N = 2000
out = shots(r, np.array([0, 0, 1]), N)
k = np.arange(1, N + 1)
running = np.cumsum(out == 1) / k
p = (1 + r[2]) / 2

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(k, running, color='#5b45b0', lw=1.5, label='measured fraction of |0>')
a1.axhline(p, color='#c47a12', ls='--', label='Born rule')
a1.plot(k, p + np.sqrt(p * (1 - p) / k), ':', color='0.5', label='±1 std. dev.')
a1.plot(k, p - np.sqrt(p * (1 - p) / k), ':', color='0.5')
a1.set_xlabel('Number of shots N'); a1.set_ylabel('Fraction of |0> outcomes')
a1.set_ylim(0, 1); a1.legend(); a1.grid(alpha=0.3)

# tomography error against the number of shots per axis
Ns = np.unique(np.logspace(1, 4.5, 15).astype(int))
err = []
for n in Ns:
    trials = []
    for _ in range(40):
        est = np.array([shots(r, e, n).mean() for e in np.eye(3)])
        trials.append(np.linalg.norm(est - r))
    err.append(np.mean(trials))
a2.loglog(Ns, err, 'o-', color='#5b45b0', label='mean |r_est - r|')
a2.loglog(Ns, err[0] * np.sqrt(Ns[0] / Ns), '--', color='0.5', label='∝ 1/√N')
a2.set_xlabel('Shots per axis'); a2.set_ylabel('Tomography error'); a2.legend(); a2.grid(alpha=0.3, which='both')
plt.tight_layout(); plt.show()

est = np.array([shots(r, e, 1000).mean() for e in np.eye(3)])
print('tomography with 1000 shots per axis:', est, ' exact:', r)

## 5. Two qubits: entangled arrows shrink

Two qubits have four amplitudes $c_{ab}$ for $|ab\rangle$, with qubit A written first. Each sphere in the lab shows one qubit on its own, described by the **reduced density matrix**

$$\rho_A=\mathrm{Tr}_B\,|\psi\rangle\langle\psi|=\frac{I+\vec r_A\cdot\vec\sigma}{2}.$$

For a product state $|\vec r_A|=1$. Entanglement shortens it: with the **concurrence** $C=2|c_{00}c_{11}-c_{01}c_{10}|$, every pure two-qubit state obeys $|\vec r_A|=|\vec r_B|=\sqrt{1-C^2}$. The entanglement entropy is the binary entropy of the eigenvalues $(1\pm|\vec r_A|)/2$ of $\rho_A$.

In [ ]:
P0, P1 = np.diag([1, 0]).astype(complex), np.diag([0, 1]).astype(complex)
CNOT = np.kron(P0, I2) + np.kron(P1, X)            # A controls, B is the target
CZ = np.diag([1, 1, 1, -1]).astype(complex)
SWAP = np.array([[1, 0, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 0, 1]], dtype=complex)
H = GATES['H'][0]
on_A = lambda U: np.kron(U, I2)
on_B = lambda U: np.kron(I2, U)

def reduced_bloch(psi):
    rho = np.outer(psi, psi.conj()).reshape(2, 2, 2, 2)   # indices a, b, a', b'
    rho_A = np.trace(rho, axis1=1, axis2=3)
    rho_B = np.trace(rho, axis1=0, axis2=2)
    return (np.real([np.trace(rho_A @ s) for s in PAULI]),
            np.real([np.trace(rho_B @ s) for s in PAULI]))

def concurrence(psi):
    return 2 * abs(psi[0] * psi[3] - psi[1] * psi[2])

def entropy_bits(r_len):
    lam = np.array([(1 + r_len) / 2, (1 - r_len) / 2])
    lam = lam[lam > 1e-15]
    return max(0.0, float(-(lam * np.log2(lam)).sum()))

def correlations(psi):
    # T[i, j] = <sigma_i (x) sigma_j>
    return np.real([[psi.conj() @ np.kron(a, b) @ psi for b in PAULI] for a in PAULI])

ket00 = np.array([1, 0, 0, 0], dtype=complex)
examples = {
    '|00>': ket00,
    'H on A': on_A(H) @ ket00,
    'H on A, then CNOT (Bell)': CNOT @ on_A(H) @ ket00,
    'R_y(60 deg) on A, then CNOT': CNOT @ on_A(R(AXES['y'], np.radians(60))) @ ket00,
    '|+>|+>, then CNOT': CNOT @ on_B(H) @ on_A(H) @ ket00,
}
print(f"{'state':>30}   C       |r_A|   |r_B|   entropy   |r_A|^2 + C^2")
for name, psi in examples.items():
    rA, rB = reduced_bloch(psi)
    C = concurrence(psi)
    print(f"{name:>30}   {C:.3f}   {np.linalg.norm(rA):.3f}   {np.linalg.norm(rB):.3f}   "
          f"{entropy_bits(np.linalg.norm(rA)):.3f}     {np.linalg.norm(rA)**2 + C**2:.12f}")

# the identity |r_A|^2 + C^2 = 1 for many random pure states
v = rng.normal(size=(2000, 4)) + 1j * rng.normal(size=(2000, 4))
v /= np.linalg.norm(v, axis=1, keepdims=True)
dev = max(abs(np.linalg.norm(reduced_bloch(p)[0])**2 + concurrence(p)**2 - 1) for p in v)
print('\nmax | |r_A|^2 + C^2 - 1 | over 2000 random states:', f'{dev:.1e}')

bell = examples['H on A, then CNOT (Bell)']
print('\nBell state amplitudes:', bell)
print('correlations <sigma_i sigma_j> (rows A = x, y, z; columns B = x, y, z):\n', correlations(bell))

### Entanglement during a gate

The lab also replays two-qubit gates continuously: $\mathrm{CNOT}^s$ is a controlled $X^s$, and $\mathrm{SWAP}^s$ multiplies the singlet part of the state by $e^{i\pi s}$. The concurrence along the way shows that a CNOT on $|+\rangle|0\rangle$ builds entanglement smoothly, while SWAP on $|0\rangle|1\rangle$ is fully entangling halfway through ($\sqrt{\mathrm{SWAP}}$) and not at all at the end.

In [ ]:
s = np.linspace(0, 1, 201)
plus0 = on_A(H) @ ket00                      # |+>|0>
ket01 = np.array([0, 1, 0, 0], dtype=complex)
C_cnot = [concurrence(frac_power(CNOT, x) @ plus0) for x in s]
C_swap = [concurrence(frac_power(SWAP, x) @ ket01) for x in s]
r_swap = [np.linalg.norm(reduced_bloch(frac_power(SWAP, x) @ ket01)[0]) for x in s]

plt.figure(figsize=(7, 4))
plt.plot(s, C_cnot, color='#5b45b0', lw=2, label='C during CNOT on |+>|0>')
plt.plot(s, C_swap, color='#e07b1f', lw=2, label='C during SWAP on |0>|1>')
plt.plot(s, r_swap, '--', color='#1f6fb4', lw=1.6, label='|r_A| during SWAP')
plt.xlabel('Fraction s of the gate'); plt.ylabel('Value'); plt.ylim(0, 1.05)
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

half = frac_power(SWAP, 0.5) @ ket01
print('sqrt(SWAP)|01> =', half, ' C =', concurrence(half))

## 6. Checks

The numbers quoted in the lab's "Predict first" answers, recomputed here.

In [ ]:
# Question 1: H twice is the identity
assert np.allclose(H @ H, I2)
# Question 2: H|+> = |0>, but R_y(90 deg)|+> is |1> up to a phase
plus = ket(np.pi / 2, 0)
assert np.allclose(bloch(H @ plus), [0, 0, 1])
assert np.allclose(bloch(R(AXES['y'], np.pi / 2) @ plus), [0, 0, -1])
# Question 3: a full turn is -I, two full turns are I
assert np.allclose(R(AXES['x'], 2 * np.pi), -I2) and np.allclose(R(AXES['x'], 4 * np.pi), I2)
# Question 4: theta = 60 deg, phi = 0
r = bloch(ket(np.radians(60), 0))
assert np.isclose((1 + r[2]) / 2, 0.75) and np.isclose((1 + r[0]) / 2, (1 + np.sqrt(3) / 2) / 2)
print('100 shots along z: expect 75 +/-', np.sqrt(100 * 0.75 * 0.25))
# Question 6: the Bell state has C = 1, <XX> = +1, <YY> = -1, <ZZ> = +1
T = correlations(bell)
assert np.isclose(concurrence(bell), 1) and np.allclose(np.diag(T), [1, -1, 1])
# Question 7: |+>|+> is unchanged by CNOT
pp = on_B(H) @ on_A(H) @ ket00
assert np.allclose(CNOT @ pp, pp) and np.isclose(concurrence(CNOT @ pp), 0)
# Question 8: sqrt(SWAP)|01> = ((1+i)|01> + (1-i)|10>)/2, maximally entangled
assert np.allclose(half, [0, (1 + 1j) / 2, (1 - 1j) / 2, 0]) and np.isclose(concurrence(half), 1)
print('All checks passed.')

## Things to try

Each idea matches a "Predict first" question in the [browser lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/qubits-bloch-sphere/).

1. **Same start, different gates.** H and $R_y(90°)$ both take $|0\rangle$ to $|+\rangle$. Apply both to `ket(np.pi/2, np.pi/2)` (that is, $|+i\rangle$) and compare the Bloch vectors.
2. **A different circuit.** Change `circuit = ['H', 'T', 'H']` to `['H', 'S', 'H']` or `['H', 'Z', 'H']`. Which single gate does $HZH$ equal?
3. **Tomography on a budget.** With only 50 shots per axis, how often does the estimated arrow land outside the sphere ($|\hat r|>1$)?
4. **A singlet.** Build $(|01\rangle-|10\rangle)/\sqrt2$ from `ket00` with X, H and CNOT, and check that all three diagonal correlations are −1.
5. **Partial entanglement.** Replace 60° in `R_y(60 deg) on A, then CNOT` with other angles. Show that $C=|\sin\alpha|$ and $|\vec r_A|=|\cos\alpha|$.